# Rocket Science & Orbital Mechanics
*From propulsion physics to interplanetary maneuvers — with live simulations*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

MU  = 3.986004418e14   # m^3/s^2
R_E = 6.371e6          # m
G0  = 9.80665          # m/s^2

rows = [
    ('mu  GM(Earth)',    f'{MU:.4e}',                    'm^3/s^2'),
    ('R_Earth',         f'{R_E/1e3:.1f}',               'km'),
    ('g0',              f'{G0:.5f}',                    'm/s^2'),
    ('v_esc (surface)', f'{np.sqrt(2*MU/R_E)/1e3:.3f}', 'km/s'),
    ('LEO alt (ISS)',   '408',                           'km'),
    ('GEO altitude',    '35 786',                       'km'),
]
print(f"{'Constant':<24} {'Value':>14}  Unit")
print('-' * 48)
for name, val, unit in rows:
    print(f'{name:<24} {val:>14}  {unit}')

Constant                          Value  Unit
------------------------------------------------
mu  GM(Earth)                3.9860e+14  m^3/s^2
R_Earth                          6371.0  km
g0                              9.80665  m/s^2
v_esc (surface)                  11.186  km/s
LEO alt (ISS)                       408  km
GEO altitude                     35 786  km


## The Tsiolkovsky Rocket Equation

A rocket accelerates by expelling mass rearward. Integrating momentum conservation over a continuous burn:

$$\Delta v = I_{sp}\, g_0 \ln\!\left(\frac{m_0}{m_f}\right)$$

| Symbol | Meaning |
|:--|:--|
| $I_{sp}$ | **Specific impulse** (s) — thrust per unit propellant weight flow; the engine efficiency figure-of-merit |
| $m_0 / m_f$ | **Mass ratio** — launch mass over final (dry) mass |
| $\Delta v$ | Total velocity increment available for all maneuvers |

The logarithm is the key: doubling $\Delta v$ requires *squaring* the mass ratio. **Staging** multiplies effectiveness by discarding empty structure between burns. High-$I_{sp}$ ion engines (1500–10 000 s) are propellant-efficient but thrust-limited; chemical engines (250–460 s) deliver large thrust at lower efficiency.

In [ ]:
@interact(
    Isp=IntSlider(value=311, min=200, max=5000, step=10,
                  description='Isp (s)', style={'description_width':'80px'},
                  continuous_update=False)
)
def rocket_equation(Isp):
    plt.close('all')
    mr = np.linspace(1.01, 25, 600)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    specs = [(280, 'Solid',    '#e74c3c'),
             (311, 'Kerolox',  '#e67e22'),
             (453, 'Hydrolox', '#2980b9'),
             (3000,'Ion',      '#27ae60')]
    for isp, lbl, c in specs:
        ax1.plot(mr, isp*G0*np.log(mr)/1e3, color=c, lw=1.5, label=f'{lbl} ({isp} s)', alpha=0.7)
    ax1.plot(mr, Isp*G0*np.log(mr)/1e3, 'k', lw=2.5, label=f'Selected ({Isp} s)')

    for lbl, dv in [('LEO+losses', 9.4), ('GTO', 12.1), ('Lunar TLI', 13.7)]:
        ax1.axhline(dv, ls=':', color='gray', alpha=0.55)
        ax1.text(24.6, dv + 0.2, lbl, ha='right', fontsize=8, color='gray')

    ax1.set(xlabel='Mass ratio  m\u2080 / mf', ylabel='\u0394v (km/s)',
            title='\u0394v vs Mass Ratio', xlim=(1, 25))
    ax1.legend(fontsize=8.5)

    dv_r = np.linspace(0, 20, 400)
    pf   = 1 - np.exp(-dv_r * 1e3 / (Isp * G0))
    ax2.plot(dv_r, pf * 100, color='#8e44ad', lw=2.5)
    ax2.fill_between(dv_r, pf * 100, alpha=0.15, color='#8e44ad')

    for lbl, dv in [('LEO', 9.4), ('GEO', 12.1), ('TLI', 13.7)]:
        pf_pt = (1 - np.exp(-dv * 1e3 / (Isp * G0))) * 100
        if pf_pt < 99:
            ax2.plot(dv, pf_pt, 'o', ms=7, color='#8e44ad')
            ax2.text(dv + 0.3, pf_pt - 6, f'{lbl}: {pf_pt:.1f}%', fontsize=8.5)

    ax2.set(xlabel='\u0394v (km/s)', ylabel='Propellant fraction (%)',
            title=f'Propellant Required  (Isp = {Isp} s)', ylim=(0, 100))
    plt.tight_layout(); plt.show()

interactive(children=(IntSlider(value=311, continuous_update=False, description='Isp (s)', max=5000, min=200, …

## Circular Orbits & the Vis-Viva Equation

For a circular orbit at altitude $h$ above Earth (orbital radius $r = R_\oplus + h$):

$$v_\text{circ} = \sqrt{\frac{\mu}{r}}, \qquad T = 2\pi\sqrt{\frac{r^3}{\mu}}, \qquad v_\text{esc} = \sqrt{2}\,v_\text{circ}$$

The **vis-viva equation** covers any conic section (circle, ellipse, hyperbola):

$$v^2 = \mu\!\left(\frac{2}{r} - \frac{1}{a}\right)$$

where $a$ is the semi-major axis. For a circular orbit $a = r$ recovers $v_\text{circ}$.  
**Key insight:** higher orbits are *slower* and have *more* (less negative) total energy $\mathcal{E} = -\mu/(2a)$. Escape corresponds to $a \to \infty$, $\mathcal{E} = 0$.

In [ ]:
@interact(
    h_max=IntSlider(value=42000, min=500, max=100000, step=500,
                    description='Alt max (km)', style={'description_width':'110px'},
                    continuous_update=False)
)
def circular_orbits(h_max):
    plt.close('all')
    h = np.linspace(10e3, h_max * 1e3, 600)
    r = R_E + h
    v_c = np.sqrt(MU / r) / 1e3
    v_e = np.sqrt(2 * MU / r) / 1e3
    T   = 2 * np.pi * np.sqrt(r**3 / MU) / 3600

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    ax1.plot(h/1e6, v_c, '#2980b9', lw=2, label='v_circ')
    ax1.plot(h/1e6, v_e, '#e74c3c', lw=2, ls='--', label='v_esc = sqrt(2) v_circ')
    ax1.fill_between(h/1e6, v_c, v_e, alpha=0.1, color='#8e44ad')

    landmarks = [('ISS  408 km', 408e3), ('GPS  20 200 km', 20200e3), ('GEO  35 786 km', 35786e3)]
    for name, alt in landmarks:
        if alt < h_max * 1e3:
            r_lm = R_E + alt
            vc_lm = np.sqrt(MU / r_lm) / 1e3
            ax1.plot(alt/1e6, vc_lm, 'ko', ms=5, zorder=5)
            ax1.annotate(name, (alt/1e6, vc_lm), textcoords='offset points',
                         xytext=(5, 4), fontsize=8)

    ax1.set(xlabel='Altitude (Mm)', ylabel='Velocity (km/s)',
            title='Orbital & Escape Velocity vs Altitude')
    ax1.legend()

    ax2.plot(h/1e6, T, '#27ae60', lw=2)
    for name, alt in [('ISS', 408e3), ('GEO', 35786e3)]:
        if alt < h_max * 1e3:
            T_lm = 2*np.pi*np.sqrt((R_E+alt)**3/MU)/3600
            ax2.plot(alt/1e6, T_lm, 'ko', ms=5, zorder=5)
            ax2.annotate(name, (alt/1e6, T_lm), textcoords='offset points',
                         xytext=(5, 4), fontsize=8)

    ax2.set(xlabel='Altitude (Mm)', ylabel='Orbital period (hours)',
            title='Orbital Period vs Altitude')
    plt.tight_layout(); plt.show()

interactive(children=(IntSlider(value=42000, continuous_update=False, description='Alt max (km)', max=100000, …

## Hohmann Transfer

The most propellant-efficient two-burn maneuver connecting two coplanar circular orbits.  
The **transfer ellipse** has periapsis at $r_1$ and apoapsis at $r_2$, so $a_t = (r_1+r_2)/2$.

**Burn 1** — prograde at periapsis; raises apoapsis from $r_1$ to $r_2$:

$$\Delta v_1 = \sqrt{\frac{\mu}{r_1}}\!\left(\sqrt{\frac{2r_2}{r_1+r_2}} - 1\right)$$

**Burn 2** — prograde at apoapsis; circularises the orbit at $r_2$:

$$\Delta v_2 = \sqrt{\frac{\mu}{r_2}}\!\left(1 - \sqrt{\frac{2r_1}{r_1+r_2}}\right)$$

**Transfer time** (half the ellipse period):

$$\tau = \pi\sqrt{\frac{a_t^3}{\mu}}$$

Both $\Delta v$ burns are *retrograde* for a descent. The Hohmann transfer is globally optimal for $r_2/r_1 \lesssim 11.94$; above that threshold a bi-elliptic transfer saves propellant.

In [4]:
def hohmann(r1, r2):
    a_t = (r1 + r2) / 2
    vt1 = np.sqrt(MU * (2/r1 - 1/a_t))
    vt2 = np.sqrt(MU * (2/r2 - 1/a_t))
    return {
        'a_t': a_t,
        'b_t': np.sqrt(r1 * r2),
        'cx' : (r1 - r2) / 2,
        'dv1': vt1 - np.sqrt(MU/r1),
        'dv2': np.sqrt(MU/r2) - vt2,
        'dv' : (vt1 - np.sqrt(MU/r1)) + (np.sqrt(MU/r2) - vt2),
        'TOF': np.pi * np.sqrt(a_t**3 / MU),
        'ecc': (r2 - r1) / (r2 + r1),
    }

@interact(
    h1=IntSlider(value=400,   min=200,  max=2000,  step=100,
                 description='h\u2081 (km)', style={'description_width':'80px'},
                 continuous_update=False),
    h2=IntSlider(value=35786, min=1000, max=80000, step=200,
                 description='h\u2082 (km)', style={'description_width':'80px'},
                 continuous_update=False),
)
def plot_hohmann(h1, h2):
    plt.close('all')
    r1 = R_E + h1 * 1e3
    r2 = R_E + h2 * 1e3
    if r2 <= r1:
        print("Set h\u2082 > h\u2081 for an ascending transfer.")
        return

    H     = hohmann(r1, r2)
    scale = r2 * 1.28
    theta = np.linspace(0, 2*np.pi, 500)
    t_arc = np.linspace(0, np.pi, 300)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6.5),
                                    gridspec_kw={'width_ratios': [1.15, 0.85]})

    ax1.add_patch(plt.Circle((0,0), R_E/scale, color='#1a6fad', zorder=5))

    ax1.plot(r1/scale*np.cos(theta), r1/scale*np.sin(theta),
             color='#2ecc71', lw=1.5, ls='--', label=f'Orbit 1  h={h1} km')
    ax1.plot(r2/scale*np.cos(theta), r2/scale*np.sin(theta),
             color='#e74c3c', lw=1.5, ls='--', label=f'Orbit 2  h={h2} km')

    a_t, b_t, cx = H['a_t'], H['b_t'], H['cx']
    xe = (cx + a_t*np.cos(t_arc)) / scale
    ye = b_t*np.sin(t_arc) / scale
    ax1.plot(xe, ye, color='#f39c12', lw=2.5, label='Transfer ellipse')
    ax1.plot(xe, -ye, color='#f39c12', lw=1, ls=':', alpha=0.35)

    ax1.plot(r1/scale, 0, 'o', color='#2ecc71', ms=11, zorder=7)
    ax1.plot(-r2/scale, 0, 's', color='#e74c3c', ms=11, zorder=7)

    dlen = 0.09
    ax1.annotate('', xytext=(r1/scale, 0), xy=(r1/scale, dlen),
                 arrowprops=dict(arrowstyle='->', color='#27ae60', lw=2.5))
    ax1.annotate('', xytext=(-r2/scale, 0), xy=(-r2/scale, -dlen),
                 arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2.5))

    ax1.text(r1/scale + 0.03, dlen + 0.03,
             f'\u0394v\u2081 = {H["dv1"]/1e3:.3f} km/s',
             fontsize=9, color='#27ae60', fontweight='bold')
    ax1.text(-r2/scale + 0.03, -(dlen + 0.07),
             f'\u0394v\u2082 = {H["dv2"]/1e3:.3f} km/s',
             fontsize=9, color='#c0392b', fontweight='bold')

    ax1.set_xlim(-1.12, 1.12); ax1.set_ylim(-1.12, 1.12)
    ax1.set_aspect('equal'); ax1.axis('off')
    ax1.legend(fontsize=8.5, loc='lower right')
    ax1.set_title('Hohmann Transfer — Orbital Geometry', pad=8)

    cats = ['\u0394v\u2081\n(1st burn)', '\u0394v\u2082\n(2nd burn)', 'Total \u0394v']
    vals = [H['dv1']/1e3, H['dv2']/1e3, H['dv']/1e3]
    clrs = ['#27ae60', '#c0392b', '#8e44ad']
    bars = ax2.bar(cats, vals, color=clrs, width=0.5, edgecolor='white', linewidth=1.5)
    for bar, v in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width()/2, v + 0.01*max(vals),
                 f'{v:.3f} km/s', ha='center', va='bottom', fontweight='bold', fontsize=10)

    ecc = H['ecc']
    ax2.set_ylabel('\u0394v (km/s)', fontsize=11)
    ax2.set_title(
        f'Transfer time:  {H["TOF"]/3600:.2f} h\n'
        f'a\u209c = {H["a_t"]/1e3:.0f} km   eccentricity = {ecc:.4f}',
        fontsize=9.5)
    ax2.set_ylim(0, max(vals)*1.45)
    plt.tight_layout(); plt.show()

interactive(children=(IntSlider(value=400, continuous_update=False, description='h₁ (km)', max=2000, min=200, …

## Mission Delta-V Budgets & Staging

The total $\Delta v$ from Earth's surface to a destination bundles: gravity/drag losses during ascent (~1.5 km/s), parking orbit insertion, and all subsequent orbital burns.  
Applying the rocket equation, the required initial wet mass for a given **payload** $m_p$ is:

$$m_0 = m_p \cdot e^{\Delta v_\text{total} / (I_{sp}\,g_0)}$$

**Staging** breaks this into $n$ sequential stages, each with its own mass ratio, making the exponent far smaller per stage. A two-stage rocket can reach LEO where a single stage often cannot.

In [ ]:
MISSIONS = [
    ('LEO  (400 km)',    9.4),
    ('MEO  (GPS)',      11.2),
    ('GEO',            12.1),
    ('Lunar orbit',    13.4),
    ('Lunar surface',  15.3),
    ('Mars orbit',     16.5),
    ('Mars surface',   19.0),
]

@interact(
    Isp=IntSlider(value=311, min=250, max=5000, step=10,
                  description='Isp (s)', style={'description_width':'80px'},
                  continuous_update=False),
    m_pay=FloatSlider(value=1000, min=100, max=20000, step=100,
                      description='Payload (kg)', style={'description_width':'100px'},
                      continuous_update=False),
)
def mission_budget(Isp, m_pay):
    plt.close('all')
    labels = [m[0] for m in MISSIONS]
    dv_km  = [m[1] for m in MISSIONS]
    m_wet  = [m_pay * np.exp(dv*1e3 / (Isp*G0)) for dv in dv_km]

    colors = plt.cm.plasma(np.linspace(0.12, 0.88, len(MISSIONS)))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

    y = np.arange(len(MISSIONS))
    ax1.barh(y, dv_km, color=colors, alpha=0.88, edgecolor='white')
    ax1.set_yticks(y); ax1.set_yticklabels(labels, fontsize=9.5)
    ax1.set_xlabel('\u0394v from ground (km/s)')
    ax1.set_title('Cumulative \u0394v Budget')
    for i, dv in enumerate(dv_km):
        ax1.text(dv + 0.1, i, f'{dv:.1f}', va='center', fontsize=8.5)
    ax1.set_xlim(0, max(dv_km)*1.12)

    ax2.barh(y, [w/1e3 for w in m_wet], color=colors, alpha=0.88, edgecolor='white')
    ax2.set_yticks(y); ax2.set_yticklabels(labels, fontsize=9.5)
    ax2.set_xlabel('Initial wet mass (tonnes)')
    ax2.set_title(f'Launch Mass for {m_pay:.0f} kg payload  (Isp={Isp} s)')
    for i, w in enumerate(m_wet):
        ax2.text(w/1e3 + max(m_wet)*0.003/1e3, i,
                 f'{w/1e3:.1f} t' if w < 1e6 else f'{w/1e6:.1f} kt',
                 va='center', fontsize=8.5)
    ax2.set_xlim(0, max(m_wet)/1e3 * 1.18)

    plt.tight_layout(); plt.show()

interactive(children=(IntSlider(value=311, continuous_update=False, description='Isp (s)', max=5000, min=250, …